In [1]:
import pandas as pd
import numpy as np
import ast
import json
import pycountry
from eu_cartel_innov.preproc import bulk_preproc

In [2]:
# load data
cartel_data = pd.read_excel('../data/raw/manual/cartel_manual_v5.xlsx', sheet_name='cartel_data')
firm_data = pd.read_excel('../data/raw/manual/cartel_manual_v5.xlsx', sheet_name='firm_data')

cartel_raw = cartel_data.copy()
firm_raw = firm_data.copy()

## clean cartel df

In [3]:
# standardize column names
def clean_colnames(df):
    df.columns = (
        df.columns.str.strip()
                  .str.lower()
                  .str.replace(r'[ .]+', '_', regex=True)
    )
    return df

cartel_data = clean_colnames(cartel_data)
firm_data = clean_colnames(firm_data)

In [4]:
# parse cartel data list columns
list_columns = ['firms_list', 'nace_code', 'nace_name', 'nace_rev', 'legal_basis']

for col in list_columns:
    cartel_data[col] = cartel_data[col].apply(
        lambda x: ast.literal_eval(x) if isinstance(x, str) else x
    )

In [5]:
# convert to datetime
cartel_date_cols = ['start_procedings_date', 'decision_date', 'start_date', 'end_date']

for col in cartel_date_cols:
    if col in cartel_data.columns:
        cartel_data[col] = pd.to_datetime(cartel_data[col], errors='coerce', dayfirst=False)

firm_date_cols = ['start_date', 'end_date']

for col in firm_date_cols:
    if col in cartel_data.columns:
        cartel_data[col] = pd.to_datetime(cartel_data[col], errors='coerce', dayfirst=False)

# convert numeric
cartel_data['fine_total'] = pd.to_numeric(cartel_data['fine_total'], errors='coerce')

In [ ]:
cartel_data.rename(columns={'decision_type': 'cartel_decision', 'fine_total': 'cartel_fine'}, inplace=True)
cartel_data['cartel_start_year'] = cartel_data['start_date'].dt.year
cartel_data['cartel_end_year'] = cartel_data['end_date'].dt.year

# duration
cartel_data['cartel_duration'] = (
    cartel_data['cartel_end_year'] - cartel_data['cartel_start_year'] + 1
)

# fine
cartel_data['log_cartel_fine'] = np.log1p(cartel_data['cartel_fine'])

### filter cartel df

In [9]:
# get all closed infringment cases (excluding financial sector) 
clean_cartel_data = cartel_data[
    (cartel_data['case_status'] == 'closed') &
    (cartel_data['cartel_decision'] != 'closure')
]

## clean firm data

In [10]:
# expand cartel firm list
cartel_exploded = clean_cartel_data.explode('firms_list').rename(columns={'firms_list': 'firm'})

# replace missing firm values for merge
cartel_exploded['firm'] = cartel_exploded['firm'].fillna('missing')
firm_data['firm'] = firm_data['firm'].fillna('missing')

In [11]:
merged_cartel_df = pd.merge(cartel_exploded, firm_data, on=['cartel_id', 'firm'], how='inner', suffixes=('_cartel','_firm'))
merged_cartel_df.replace('missing', np.nan, inplace=True)

In [12]:
clean_firm_data = merged_cartel_df[[
    'cartel_id',
    'cartel_name_cartel',
    'firm',
    'address',
    'city',
    'country',
    'start_date_firm',
    'date_estimated_firm',
    'end_date_firm',
    'date_estimated_1_firm'
]].rename(columns={
    'cartel_name_cartel': 'cartel_name',
    'firm' : 'firm_name',
    'start_date_firm': 'start_date',
    'date_estimated_firm': 'stat_date_est',
    'end_date_firm': 'end_date',
    'date_estimated.1_firm': 'end_date_est', 
})

In [13]:
clean_firm_data = bulk_preproc(clean_firm_data, 'firm_name', keep_spaces=False)
print(clean_firm_data['firm_name'].nunique())
print(clean_firm_data['firm_name_preproc'].nunique())

preprocessing 1,312 unique values out of 1,603 total rows...
created column: 'firm_name_preproc'
1312
1301


In [14]:
# get all cases from 1998 onwards
clean_firm_data = clean_firm_data[(clean_firm_data['start_date'] >= '1998-01-01')]
print(clean_firm_data['firm_name'].nunique())
print(clean_firm_data['firm_name_preproc'].nunique())

756
750


In [15]:
def country_name_to_iso(name):
    try:
        return pycountry.countries.lookup(name).alpha_2
    except LookupError:
        return None

clean_firm_data['ctry_code'] = clean_firm_data['country'].apply(country_name_to_iso)
clean_firm_data.loc[clean_firm_data['country'] == 'Korea', 'ctry_code'] = 'KR'

In [16]:
# get unique treated firms list 
treated_firms = clean_firm_data['firm_name'].unique().tolist()

# save json file
with open('../data/processed/treat_firm_list.json', 'w') as f:
    json.dump(treated_firms, f)

In [17]:
clean_firm_data.to_excel('../data/processed/clean_firm_data.xlsx', index=False)
clean_cartel_data.to_excel('../data/processed/clean_cartel_data.xlsx', index=False)